<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase54.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 54 — authorized companion-source discovery and joinability audit

Phase 53 created a deterministic private FHIR R4 source representation with:

- 258 `Patient` resources;
- 4,128 `Observation` resources;
- exact source→FHIR person/measurement/value/timepoint preservation;
- explicit non-invention of units, dates, acquisition IDs, model versions, provenance, and diagnosis.

The remaining official-target problem is not representation. It is **missing source evidence**.

Phase 54 searches for authorized companion files that can legitimately supply those missing dimensions and tests whether they are actually joinable to the 258-person longitudinal source.

### Evidence dimensions searched

1. demographics needed for an honest `PERSON` mapping;
2. explicit measurement-unit documentation;
3. acquisition / image / study / series identifiers;
4. scan date / modality / procedure-like source evidence;
5. extraction software / model / pipeline version;
6. provenance / processing identifiers;
7. clinical diagnosis / MCI source status.

### Joinability rules

- A companion file is not useful merely because it contains a promising column name.
- Person-level evidence must overlap the real source person keys.
- Acquisition evidence should also carry a visit/timepoint/date bridge when possible.
- No identifier values, participant IDs, exact private paths, or row-level source data are exported publicly.
- Companion tables are read at row level only after an explicit private authorization-scope attestation.


In [1]:
# CODE CELL 1/10 — Drive, constants, helpers
import os, re, csv, json, gzip, hashlib, tempfile, shutil, zipfile, time, math
import datetime as dt
from pathlib import Path
from collections import defaultdict, Counter

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='54_authorized_companion_source_discovery_and_joinability_audit'

EXPECTED_P53_SHAREABLE='4ff47277e7e3504c98659b2bab6c86dc31324bfda286b0cfc4d151b0b0fadc16'
EXPECTED_P53_COUNTS='be5b97728c2a6ec9740a8ab35d98369c01ea5dacd73d0ec352f099add3dc96fb'
SOURCE_TOKEN='069caf1537e082c6'
EXPECTED_PERSONS=258
EXPECTED_OBS=4128

SUPPORTED={'.csv','.tsv','.txt','.gz','.parquet','.xlsx','.xls','.json'}
MAX_NEARBY_FILES=int(os.environ.get('PHASE54_MAX_NEARBY_FILES','250'))
MAX_HEADER_CANDIDATES=int(os.environ.get('PHASE54_MAX_HEADER_CANDIDATES','60'))
MAX_ROW_PROFILE_CANDIDATES=int(os.environ.get('PHASE54_MAX_ROW_PROFILE_CANDIDATES','18'))
MAX_PROFILE_BYTES=int(os.environ.get('PHASE54_MAX_PROFILE_BYTES',str(800_000_000)))
SCAN_TIME_LIMIT=int(os.environ.get('PHASE54_SCAN_TIME_LIMIT_SECONDS','150'))

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def path_token(p):
    return hashlib.sha256(str(Path(p).resolve()).encode('utf-8')).hexdigest()[:16]

def value_token(v):
    return hashlib.sha256(str(v).encode('utf-8')).hexdigest()[:16]

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader();w.writerows(rows)

if os.environ.get('PHASE54_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE54_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir():ROOTS.append(sd)

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase54_'))
STAGE=WORK/'public_outputs';STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE54_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase54')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

PRIVATE_ROOT=MYDRIVE/'ICHI2027'/'_PRIVATE_REAL_MRI'
PRIVATE54=PRIVATE_ROOT/'Phase54'/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
PRIVATE54.mkdir(parents=True,exist_ok=False)

print('Phase54 public output:',OUT)
print('Private companion-audit workspace prepared.')


Mounted at /content/drive
Phase54 public output: /content/drive/MyDrive/ICHI2027/Phase54/run_20261001T054919_003892Z
Private companion-audit workspace prepared.


In [2]:
# CODE CELL 2/10 — Verify exact Phase53 public/private chain

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

shareables=[]
counts_files=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            p=Path(here)/fn
            try:
                if name_matches(fn,'ICHI2027_Phase53_SHAREABLE','.json') and sha256(p)==EXPECTED_P53_SHAREABLE:
                    shareables.append(p)
                elif name_matches(fn,'ICHI2027_Phase53_FHIR_RESOURCE_COUNTS','.csv') and sha256(p)==EXPECTED_P53_COUNTS:
                    counts_files.append(p)
            except OSError:
                pass

if not shareables: raise FileNotFoundError('Exact Phase53 SHAREABLE.json not found.')
if not counts_files: raise FileNotFoundError('Exact Phase53 FHIR resource counts CSV not found.')

P53=sorted(shareables,key=str)[0]
C53=sorted(counts_files,key=str)[0]
p53=json.loads(P53.read_text('utf-8'))

assert p53['phase']=='53_authorized_real_source_to_private_fhir_r4_bulk_ndjson'
assert p53['authorization_attestation_carried_forward'] is True
assert p53['fhir_patient_count']==EXPECTED_PERSONS
assert p53['fhir_observation_count']==EXPECTED_OBS
assert p53['fhir_imagingstudy_count']==0
assert p53['fhir_device_count']==0
assert p53['fhir_provenance_count']==0
assert p53['fhir_condition_count']==0
assert p53['ready_for_official_armA'] is False
assert p53['ready_for_private_fhir_to_target_preflight'] is True

# Find the matching private Phase53 FHIR manifest.
p53_private=[]
phase53_private_root=PRIVATE_ROOT/'Phase53'
if phase53_private_root.is_dir():
    for run in sorted(phase53_private_root.glob('run_*'),reverse=True):
        mf=run/'ICHI2027_Phase53_PRIVATE_FHIR_MANIFEST.json'
        if not mf.is_file():continue
        try:
            m=json.loads(mf.read_text('utf-8'))
            if (
                m.get('source_file_token')==SOURCE_TOKEN
                and int(m.get('patient_count',-1))==EXPECTED_PERSONS
                and int(m.get('observation_count',-1))==EXPECTED_OBS
                and m.get('structural_validation_all_passed') is True
            ):
                p53_private.append((run,mf,m))
        except Exception:
            pass

if not p53_private:
    raise FileNotFoundError('Matching private Phase53 FHIR manifest not found.')

P53_PRIVATE_RUN,P53_PRIVATE_MANIFEST,private53=p53_private[0]

# Follow the frozen private chain back to the Phase52 source manifest/normalized source.
P52_PRIVATE_MANIFEST=Path(private53['phase52_private_manifest_path'])
NORMALIZED=Path(private53['phase52_normalized_source_path'])
if not P52_PRIVATE_MANIFEST.is_file():raise FileNotFoundError(P52_PRIVATE_MANIFEST)
if not NORMALIZED.is_file():raise FileNotFoundError(NORMALIZED)

private52=json.loads(P52_PRIVATE_MANIFEST.read_text('utf-8'))
SOURCE_PATH=Path(private52['exact_private_path'])
PERSON_COL=private52['person_key_column']
if not SOURCE_PATH.is_file():raise FileNotFoundError(SOURCE_PATH)

print('PASS: Phase53 public/private chain verified.')
print('Source directory anchored privately for targeted companion search.')


PASS: Phase53 public/private chain verified.
Source directory anchored privately for targeted companion search.


In [3]:
# CODE CELL 3/10 — Explicit companion-source authorization-scope attestation

AUTH_FILE=PRIVATE_ROOT/'AUTHORIZED_COMPANION_SOURCE_ATTESTATION.json'

existing=False
auth={}
if AUTH_FILE.exists():
    try:
        auth=json.loads(AUTH_FILE.read_text('utf-8'))
        day=dt.date.fromisoformat(str(auth['attestation_date_utc']))
        existing=(
            day<=dt.datetime.now(dt.timezone.utc).date()
            and auth.get('same_governing_authorization_as_selected_source') is True
            and auth.get('authorized_for_local_join_analysis') is True
            and auth.get('no_public_participant_level_release') is True
            and bool(str(auth.get('attested_by','')).strip())
        )
    except Exception:
        existing=False

if existing:
    print('Existing private companion-source authorization attestation is valid.')
else:
    print('Companion tables may contain separate participant-level source information.')
    print('Answer YES only if these nearby/source-family files are governed by the same authorized access/data-use terms.')
    ans1=input('The companion tables I am allowing Phase54 to inspect are covered by the same governing authorization [yes/no]: ').strip().lower()
    ans2=input('Those terms allow local linkage/join analysis for this research [yes/no]: ').strip().lower()
    ans3=input('I will not place participant-level source data or identifiers in public outputs [yes/no]: ').strip().lower()
    who=input('Enter your name or initials for this PRIVATE attestation: ').strip()

    auth={
      'same_governing_authorization_as_selected_source':ans1 in {'yes','y'},
      'authorized_for_local_join_analysis':ans2 in {'yes','y'},
      'no_public_participant_level_release':ans3 in {'yes','y'},
      'attested_by':who,
      'attestation_date_utc':dt.datetime.now(dt.timezone.utc).date().isoformat(),
      'source_anchor_token':SOURCE_TOKEN,
      'notes':'Local investigator attestation; does not replace governing data-use terms.'
    }
    AUTH_FILE.write_text(json.dumps(auth,indent=2)+'\n',encoding='utf-8')

companion_auth_valid=(
    auth.get('same_governing_authorization_as_selected_source') is True
    and auth.get('authorized_for_local_join_analysis') is True
    and auth.get('no_public_participant_level_release') is True
    and bool(str(auth.get('attested_by','')).strip())
)

if not companion_auth_valid:
    raise RuntimeError('Companion-source authorization scope not affirmed. Phase54 stops before row-level linkage.')

print('PASS: companion-source authorization scope attested.')


Companion tables may contain separate participant-level source information.
Answer YES only if these nearby/source-family files are governed by the same authorized access/data-use terms.
The companion tables I am allowing Phase54 to inspect are covered by the same governing authorization [yes/no]: YES
Those terms allow local linkage/join analysis for this research [yes/no]: YES
I will not place participant-level source data or identifiers in public outputs [yes/no]: YES
Enter your name or initials for this PRIVATE attestation: SB
PASS: companion-source authorization scope attested.


In [4]:
# CODE CELL 4/10 — Read main source person/timepoint keys privately

import pandas as pd
import numpy as np

def load_table(p,usecols=None,nrows=None):
    suf=p.suffix.lower()
    if suf=='.gz':return pd.read_csv(p,compression='gzip',usecols=usecols,nrows=nrows,low_memory=False)
    if suf=='.csv':return pd.read_csv(p,usecols=usecols,nrows=nrows,low_memory=False)
    if suf=='.tsv':return pd.read_csv(p,sep='\t',usecols=usecols,nrows=nrows,low_memory=False)
    if suf=='.txt':
        try:return pd.read_csv(p,sep='\t',usecols=usecols,nrows=nrows,low_memory=False)
        except Exception:return pd.read_csv(p,usecols=usecols,nrows=nrows,low_memory=False)
    if suf=='.parquet':
        df=pd.read_parquet(p,columns=usecols)
        return df if nrows is None else df.head(nrows)
    if suf in {'.xlsx','.xls'}:
        return pd.read_excel(p,usecols=usecols,nrows=nrows)
    if suf=='.json':
        return pd.read_json(p)
    raise ValueError(suf)

def norm(c):
    return re.sub(r'[^a-z0-9]+','_',str(c).strip().lower()).strip('_')

def visit_class(v):
    s=str(v).strip().lower().replace(' ','').replace('_','').replace('-','')
    if s in {'bl','baseline','m0','m00','month0','v0','visit0'}:return 'BASELINE'
    if s in {'m12','month12','12m','12month','12months','v12','visit12','yr1','year1'}:return 'M12'
    m=re.search(r'(?:m|month|v|visit)(\d+)',s)
    if m:
        mo=int(m.group(1))
        if mo==0:return 'BASELINE'
        if 9<=mo<=15:return 'M12'
    return ''

main=load_table(SOURCE_PATH)
if PERSON_COL not in main.columns:raise RuntimeError('Frozen Phase52 person-key column no longer present.')

main_people=set(main[PERSON_COL].dropna().astype(str))
assert len(main_people)==EXPECTED_PERSONS

# Main source is wide; every source person has baseline/M12 measurement columns.
main_person_time={(p,tp) for p in main_people for tp in ('BASELINE','M12')}

print('PASS: main-source join universe loaded privately.')
print('Main source persons:',len(main_people))


PASS: main-source join universe loaded privately.
Main source persons: 258


In [5]:
# CODE CELL 5/10 — Targeted nearby companion-file discovery

# Search same directory first, then parent/siblings up to two levels.
anchor_dir=SOURCE_PATH.parent
search_dirs=[anchor_dir]
if anchor_dir.parent!=anchor_dir:
    search_dirs.append(anchor_dir.parent)
    try:
        search_dirs.extend([p for p in anchor_dir.parent.iterdir() if p.is_dir()][:80])
    except Exception:
        pass

# De-duplicate and exclude generated ICHI phase/private outputs except source-family files.
unique_dirs=[]
seen=set()
for d in search_dirs:
    try:k=str(d.resolve())
    except:k=str(d)
    if k in seen:continue
    seen.add(k)
    if str(PRIVATE_ROOT).lower() in k.lower():continue
    unique_dirs.append(d)

SEMANTIC_PATTERNS={
 'person_key':['rid','ptid','subject','subject_id','participant','participant_id','person_id','patient_id'],
 'visit':['viscode','viscode2','visit','visit_code','timepoint','time_point'],
 'date':['examdate','exam_date','scan_date','scandate','acq_date','acquisition_date','study_date'],
 'gender':['gender','sex','ptgender'],
 'birth':['birthdate','birth_date','dob','year_of_birth','birthyear','ptdob','ptbirth'],
 'race':['race','ptraccat'],
 'ethnicity':['ethnicity','ptethcat'],
 'diagnosis':['dx','diagnosis','diagnostic','research_group','group'],
 'unit':['unit','units','ucum'],
 'study_uid':['studyinstanceuid','study_instance_uid','studyuid','study_uid'],
 'series_uid':['seriesinstanceuid','series_instance_uid','seriesuid','series_uid'],
 'image_uid':['imageuid','image_uid','sopinstanceuid','sop_instance_uid','image_id'],
 'modality':['modality','mr_modality','scan_type'],
 'procedure':['procedure','exam_type','study_description','series_description'],
 'model_version':['freesurfer_version','software_version','model_version','algorithm_version','pipeline_version','version'],
 'provenance':['provenance','processing_id','process_id','pipeline','software','device','algorithm']
}

def header_only(p):
    try:
        suf=p.suffix.lower()
        if suf=='.gz':
            with gzip.open(p,'rt',encoding='utf-8-sig',errors='replace') as f:
                line=f.readline()
                sep='\t' if line.count('\t')>line.count(',') else ','
                return next(csv.reader([line],delimiter=sep))
        if suf in {'.csv','.tsv','.txt'}:
            with p.open('r',encoding='utf-8-sig',errors='replace') as f:
                line=f.readline()
                sep='\t' if (suf=='.tsv' or line.count('\t')>line.count(',')) else ','
                return next(csv.reader([line],delimiter=sep))
        if suf=='.parquet':
            import pyarrow.parquet as pq
            return pq.ParquetFile(p).schema.names
        if suf in {'.xlsx','.xls'}:
            if p.stat().st_size>250_000_000:return []
            return list(pd.read_excel(p,nrows=0).columns)
        return []
    except Exception:
        return []

start=time.monotonic()
raw_files=[]
for d in unique_dirs:
    if time.monotonic()-start>SCAN_TIME_LIMIT:break
    try:
        for here,dirs,files in os.walk(d,topdown=True,followlinks=False):
            rel_depth=len(Path(here).parts)-len(Path(d).parts)
            if rel_depth>2:
                dirs[:]=[]
                continue
            dirs[:]=[
                x for x in dirs
                if x not in {'.git','node_modules','.ipynb_checkpoints'}
                and not x.startswith('.Trash')
            ]
            for fn in files:
                if len(raw_files)>=MAX_NEARBY_FILES:break
                p=Path(here)/fn
                if p==SOURCE_PATH:continue
                if p.suffix.lower() not in SUPPORTED:continue
                if re.match(r'^(ICHI2027_|Phase\d+)',fn,re.I):continue
                try:
                    size=p.stat().st_size
                except OSError:
                    continue
                if size<=0:continue
                raw_files.append((p,size))
            if len(raw_files)>=MAX_NEARBY_FILES:break
        if len(raw_files)>=MAX_NEARBY_FILES:break
    except (OSError,PermissionError):
        continue

candidate_rows=[]
private_candidate_paths={}
for p,size in raw_files[:MAX_HEADER_CANDIDATES]:
    cols=header_only(p)
    ncols=[norm(c) for c in cols]
    found={}
    for dim,pats in SEMANTIC_PATTERNS.items():
        found[dim]=sum(any(pat==n or pat in n for pat in pats) for n in ncols)
    semantic_dimensions=sum(v>0 for v in found.values())
    # Keep files with a person key plus at least one missing-evidence dimension,
    # or documentation-like files with unit/model/provenance semantics.
    useful=(
      found['person_key']>0 and semantic_dimensions>=2
    ) or (
      found['unit']>0 or found['model_version']>0 or found['provenance']>0
    )
    if not useful:continue
    tok=path_token(p)
    row={
      'file_token':tok,
      'extension':p.suffix.lower(),
      'bytes':size,
      'header_columns':len(cols),
      'semantic_dimensions_detected':semantic_dimensions,
      'private_path_exported':False
    }
    for dim in SEMANTIC_PATTERNS:
        row[f'{dim}_header_hits']=found[dim]
    candidate_rows.append(row)
    private_candidate_paths[tok]=p

candidate_rows=sorted(
    candidate_rows,
    key=lambda r:(-r['semantic_dimensions_detected'],-r['person_key_header_hits'],r['bytes'])
)

print('Nearby files inspected:',len(raw_files))
print('Companion header candidates retained:',len(candidate_rows))
print('Discovery seconds:',round(time.monotonic()-start,2))


Nearby files inspected: 20
Companion header candidates retained: 0
Discovery seconds: 7.13


In [6]:
# CODE CELL 6/10 — Row-level joinability audit for top candidates

def choose_cols(cols,patterns):
    out=[]
    for c in cols:
        n=norm(c)
        if any(p==n or p in n for p in patterns):
            out.append(c)
    return out

join_rows=[]
for cmeta in candidate_rows[:MAX_ROW_PROFILE_CANDIDATES]:
    tok=cmeta['file_token']
    p=private_candidate_paths[tok]
    if cmeta['bytes']>MAX_PROFILE_BYTES:
        join_rows.append({
          'file_token':tok,'profile_status':'SKIPPED_TOO_LARGE',
          'person_key_column_token':'','best_person_overlap':0,'best_person_overlap_fraction':0.0,
          'visit_bridge_present':False,'baseline_overlap_persons':0,'m12_overlap_persons':0,
          'person_timepoint_overlap_pairs':0,'diagnosis_overlap_persons':0,
          'acquisition_id_rows_on_main_people':0,'unit_evidence_rows_on_main_people':0,
          'model_version_rows_on_main_people':0,'provenance_rows_on_main_people':0,
          'demographic_rows_on_main_people':0,'private_values_exported':False
        })
        continue

    try:
        cols=header_only(p)
        person_cols=choose_cols(cols,SEMANTIC_PATTERNS['person_key'])
        if not person_cols:
            join_rows.append({
              'file_token':tok,'profile_status':'NO_PERSON_KEY',
              'person_key_column_token':'','best_person_overlap':0,'best_person_overlap_fraction':0.0,
              'visit_bridge_present':False,'baseline_overlap_persons':0,'m12_overlap_persons':0,
              'person_timepoint_overlap_pairs':0,'diagnosis_overlap_persons':0,
              'acquisition_id_rows_on_main_people':0,'unit_evidence_rows_on_main_people':0,
              'model_version_rows_on_main_people':0,'provenance_rows_on_main_people':0,
              'demographic_rows_on_main_people':0,'private_values_exported':False
            })
            continue

        semantic_cols=set(person_cols)
        for pats in SEMANTIC_PATTERNS.values():
            semantic_cols.update(choose_cols(cols,pats))
        semantic_cols=list(semantic_cols)

        df=load_table(p,usecols=semantic_cols)
        best_col=None;best_overlap=set()
        for pc in person_cols:
            vals=set(df[pc].dropna().astype(str))
            ov=vals & main_people
            if len(ov)>len(best_overlap):
                best_col=pc;best_overlap=ov

        if best_col is None:
            raise RuntimeError('No usable person-key column after read.')

        sub=df[df[best_col].dropna().astype(str).isin(main_people)].copy()

        visit_cols=choose_cols(cols,SEMANTIC_PATTERNS['visit'])
        date_cols=choose_cols(cols,SEMANTIC_PATTERNS['date'])
        baseline_people=set();m12_people=set();pt_pairs=set()

        if visit_cols:
            vc=visit_cols[0]
            sub['_tp']=sub[vc].map(visit_class)
            baseline_people=set(sub.loc[sub['_tp']=='BASELINE',best_col].dropna().astype(str))
            m12_people=set(sub.loc[sub['_tp']=='M12',best_col].dropna().astype(str))
            pt_pairs={(str(pid),str(tp)) for pid,tp in sub.loc[sub['_tp'].isin(['BASELINE','M12']),[best_col,'_tp']].itertuples(index=False,name=None)}
        elif date_cols:
            dc=date_cols[0]
            sub['_date']=pd.to_datetime(sub[dc],errors='coerce')
            for pid,g in sub.dropna(subset=[best_col,'_date']).groupby(best_col):
                dates=sorted(g['_date'].dt.normalize().unique())
                if not dates:continue
                first=pd.Timestamp(dates[0])
                baseline_people.add(str(pid));pt_pairs.add((str(pid),'BASELINE'))
                if any(270 <= (pd.Timestamp(x)-first).days <= 455 for x in dates[1:]):
                    m12_people.add(str(pid));pt_pairs.add((str(pid),'M12'))

        def nonempty_rows(pattern_key):
            cc=choose_cols(cols,SEMANTIC_PATTERNS[pattern_key])
            if not cc:return 0
            return int(sub[cc].notna().any(axis=1).sum())

        diag_cols=choose_cols(cols,SEMANTIC_PATTERNS['diagnosis'])
        diagnosis_people=0
        if diag_cols:
            mask=False
            for dc in diag_cols:
                this=sub[dc].astype(str).str.lower().str.contains(r'\bmci\b|mild cognitive',regex=True,na=False)
                mask=this if isinstance(mask,bool) else (mask|this)
            diagnosis_people=sub.loc[mask,best_col].dropna().astype(str).nunique()

        acquisition_cols=(choose_cols(cols,SEMANTIC_PATTERNS['study_uid'])+
                          choose_cols(cols,SEMANTIC_PATTERNS['series_uid'])+
                          choose_cols(cols,SEMANTIC_PATTERNS['image_uid']))
        acquisition_rows=int(sub[acquisition_cols].notna().any(axis=1).sum()) if acquisition_cols else 0

        unit_cols=choose_cols(cols,SEMANTIC_PATTERNS['unit'])
        unit_rows=int(sub[unit_cols].notna().any(axis=1).sum()) if unit_cols else 0

        model_cols=choose_cols(cols,SEMANTIC_PATTERNS['model_version'])
        model_rows=int(sub[model_cols].notna().any(axis=1).sum()) if model_cols else 0

        prov_cols=choose_cols(cols,SEMANTIC_PATTERNS['provenance'])
        prov_rows=int(sub[prov_cols].notna().any(axis=1).sum()) if prov_cols else 0

        demo_cols=(choose_cols(cols,SEMANTIC_PATTERNS['gender'])+
                   choose_cols(cols,SEMANTIC_PATTERNS['birth'])+
                   choose_cols(cols,SEMANTIC_PATTERNS['race'])+
                   choose_cols(cols,SEMANTIC_PATTERNS['ethnicity']))
        demo_rows=int(sub[demo_cols].notna().any(axis=1).sum()) if demo_cols else 0

        join_rows.append({
          'file_token':tok,'profile_status':'PROFILED',
          'person_key_column_token':value_token(best_col),
          'best_person_overlap':len(best_overlap),
          'best_person_overlap_fraction':len(best_overlap)/EXPECTED_PERSONS,
          'visit_bridge_present':bool(visit_cols or date_cols),
          'baseline_overlap_persons':len(baseline_people),
          'm12_overlap_persons':len(m12_people),
          'person_timepoint_overlap_pairs':len(pt_pairs & main_person_time),
          'diagnosis_overlap_persons':diagnosis_people,
          'acquisition_id_rows_on_main_people':acquisition_rows,
          'unit_evidence_rows_on_main_people':unit_rows,
          'model_version_rows_on_main_people':model_rows,
          'provenance_rows_on_main_people':prov_rows,
          'demographic_rows_on_main_people':demo_rows,
          'private_values_exported':False
        })

    except Exception as e:
        join_rows.append({
          'file_token':tok,'profile_status':'READ_OR_PROFILE_ERROR',
          'person_key_column_token':'','best_person_overlap':0,'best_person_overlap_fraction':0.0,
          'visit_bridge_present':False,'baseline_overlap_persons':0,'m12_overlap_persons':0,
          'person_timepoint_overlap_pairs':0,'diagnosis_overlap_persons':0,
          'acquisition_id_rows_on_main_people':0,'unit_evidence_rows_on_main_people':0,
          'model_version_rows_on_main_people':0,'provenance_rows_on_main_people':0,
          'demographic_rows_on_main_people':0,'private_values_exported':False
        })

print('Row-level companion candidates profiled:',sum(r['profile_status']=='PROFILED' for r in join_rows))


Row-level companion candidates profiled: 0


In [7]:
# CODE CELL 7/10 — Select best evidence source per missing dimension

profiled=[r for r in join_rows if r['profile_status']=='PROFILED']

def best_for(metric, require_timepoint=False):
    eligible=[r for r in profiled if r[metric]>0 and r['best_person_overlap']>0]
    if require_timepoint:
        eligible=[r for r in eligible if r['person_timepoint_overlap_pairs']>0]
    if not eligible:return None
    return max(
        eligible,
        key=lambda r:(r[metric],r['person_timepoint_overlap_pairs'],r['best_person_overlap'])
    )

best_demo=best_for('demographic_rows_on_main_people')
best_diag=best_for('diagnosis_overlap_persons')
best_acq=best_for('acquisition_id_rows_on_main_people',require_timepoint=True)
best_unit=best_for('unit_evidence_rows_on_main_people')
best_model=best_for('model_version_rows_on_main_people')
best_prov=best_for('provenance_rows_on_main_people')

dimension_rows=[]
for dim,best,metric,needs_tp in [
 ('demographics',best_demo,'demographic_rows_on_main_people',False),
 ('clinical_diagnosis_mci',best_diag,'diagnosis_overlap_persons',False),
 ('acquisition_image_identifiers',best_acq,'acquisition_id_rows_on_main_people',True),
 ('measurement_unit_evidence',best_unit,'unit_evidence_rows_on_main_people',False),
 ('model_software_version',best_model,'model_version_rows_on_main_people',False),
 ('provenance_processing',best_prov,'provenance_rows_on_main_people',False),
]:
    dimension_rows.append({
      'dimension':dim,
      'candidate_found':best is not None,
      'selected_file_token':best['file_token'] if best else '',
      'evidence_rows_or_persons':best[metric] if best else 0,
      'person_overlap':best['best_person_overlap'] if best else 0,
      'person_timepoint_overlap_pairs':best['person_timepoint_overlap_pairs'] if best else 0,
      'timepoint_bridge_required':needs_tp,
      'private_values_exported':False
    })

for r in dimension_rows:
    print(r['dimension'],'=>',('FOUND' if r['candidate_found'] else 'NOT FOUND'),
          'overlap',r['person_overlap'],'timepoints',r['person_timepoint_overlap_pairs'])


demographics => NOT FOUND overlap 0 timepoints 0
clinical_diagnosis_mci => NOT FOUND overlap 0 timepoints 0
acquisition_image_identifiers => NOT FOUND overlap 0 timepoints 0
measurement_unit_evidence => NOT FOUND overlap 0 timepoints 0
model_software_version => NOT FOUND overlap 0 timepoints 0
provenance_processing => NOT FOUND overlap 0 timepoints 0


In [8]:
# CODE CELL 8/10 — Updated evidence-gap closure and Arm A readiness

# A dimension is "candidate-supported" here only when a joinable source was discovered.
# It is not yet merged into FHIR/OMOP and therefore does not by itself clear official Arm A.

dim={r['dimension']:r for r in dimension_rows}

gap_rows=[
 {
  'gate':'demographics_for_person',
  'candidate_evidence_found':dim['demographics']['candidate_found'],
  'official_gate_cleared':False,
  'reason':'Companion source must still be semantically reviewed and merged under the frozen PERSON policy.'
 },
 {
  'gate':'clinical_diagnosis_mci_source',
  'candidate_evidence_found':dim['clinical_diagnosis_mci']['candidate_found'],
  'official_gate_cleared':False,
  'reason':'Diagnosis evidence must be source-defined and linked; it cannot be inferred from manuscript membership.'
 },
 {
  'gate':'real_acquisition_image_identifiers',
  'candidate_evidence_found':dim['acquisition_image_identifiers']['candidate_found'],
  'official_gate_cleared':False,
  'reason':'Real study/series/image identity must be linked at person/timepoint level before ImagingStudy/IMAGE_OCCURRENCE claims.'
 },
 {
  'gate':'explicit_measurement_units',
  'candidate_evidence_found':dim['measurement_unit_evidence']['candidate_found'],
  'official_gate_cleared':False,
  'reason':'Unit evidence must be mapped to the exact measurement fields before Quantity/OMOP unit claims.'
 },
 {
  'gate':'model_version_lineage',
  'candidate_evidence_found':dim['model_software_version']['candidate_found'],
  'official_gate_cleared':False,
  'reason':'Model/software version must be joined to the exact extraction evidence.'
 },
 {
  'gate':'provenance_processing_lineage',
  'candidate_evidence_found':dim['provenance_processing']['candidate_found'],
  'official_gate_cleared':False,
  'reason':'Processing/provenance evidence must be linked before a provenance-preservation experiment.'
 }
]

found_count=sum(r['candidate_evidence_found'] for r in gap_rows)
all_core_candidates=all(r['candidate_evidence_found'] for r in gap_rows if r['gate'] in {
    'demographics_for_person','real_acquisition_image_identifiers','explicit_measurement_units'
})

if found_count>0:
    next_step='PHASE55_REVIEW_AND_FREEZE_COMPANION_SOURCE_BINDINGS_THEN_ENRICH_PRIVATE_FHIR'
else:
    next_step='COMPANION_EVIDENCE_NOT_FOUND_IN_NEARBY_AUTHORIZED_SOURCE_FAMILY'

ready_for_official_armA=False

print('Missing-evidence dimensions with joinable candidate source:',found_count,'/',len(gap_rows))
print('Ready for official Arm A:',ready_for_official_armA)
print('Next step:',next_step)


Missing-evidence dimensions with joinable candidate source: 0 / 6
Ready for official Arm A: False
Next step: COMPANION_EVIDENCE_NOT_FOUND_IN_NEARBY_AUTHORIZED_SOURCE_FAMILY


In [9]:
# CODE CELL 9/10 — Freeze private companion audit manifest

private_manifest={
 'project':PROJECT,
 'phase':PHASE,
 'created_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'source_anchor_token':SOURCE_TOKEN,
 'source_anchor_exact_path':str(SOURCE_PATH),
 'authorization_scope_attestation':auth,
 'candidate_private_paths':{
    tok:str(path) for tok,path in private_candidate_paths.items()
 },
 'candidate_headers':{
    tok:header_only(path) for tok,path in private_candidate_paths.items()
 },
 'joinability_public_rows':join_rows,
 'selected_dimension_sources':dimension_rows,
 'notes':'Private manifest contains exact paths/headers. Public outputs contain only tokens and aggregate overlap counts.'
}

PRIVATE_MANIFEST=PRIVATE54/'ICHI2027_Phase54_PRIVATE_COMPANION_AUDIT.json'
PRIVATE_MANIFEST.write_text(json.dumps(private_manifest,indent=2,default=str)+'\n',encoding='utf-8')

print('PASS: private companion-source audit manifest frozen.')


PASS: private companion-source audit manifest frozen.


In [10]:
# CODE CELL 10/10 — Public-safe outputs and shareable summary

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

artifacts=[]
artifacts.append(save('ICHI2027_Phase54_COMPANION_CANDIDATE_INVENTORY.csv',candidate_rows if candidate_rows else [{
 'file_token':'','extension':'','bytes':0,'header_columns':0,'semantic_dimensions_detected':0,
 'private_path_exported':False,
 **{f'{d}_header_hits':0 for d in SEMANTIC_PATTERNS}
}]))
artifacts.append(save('ICHI2027_Phase54_JOINABILITY_AUDIT.csv',join_rows if join_rows else [{
 'file_token':'','profile_status':'NO_CANDIDATE','person_key_column_token':'',
 'best_person_overlap':0,'best_person_overlap_fraction':0.0,'visit_bridge_present':False,
 'baseline_overlap_persons':0,'m12_overlap_persons':0,'person_timepoint_overlap_pairs':0,
 'diagnosis_overlap_persons':0,'acquisition_id_rows_on_main_people':0,
 'unit_evidence_rows_on_main_people':0,'model_version_rows_on_main_people':0,
 'provenance_rows_on_main_people':0,'demographic_rows_on_main_people':0,
 'private_values_exported':False
}]))
artifacts.append(save('ICHI2027_Phase54_SELECTED_DIMENSION_SOURCES.csv',dimension_rows))
artifacts.append(save('ICHI2027_Phase54_ARM_A_GAP_UPDATE.csv',gap_rows))

readme=STAGE/'ICHI2027_Phase54_README.txt'
readme.write_text(
 'Phase54 searches only a bounded, nearby authorized source family anchored to the exact Phase52/53 source location.\n'
 'A companion file is counted as evidence only when its person key overlaps the 258-person source; acquisition evidence additionally requires a visit/timepoint/date bridge when possible.\n'
 'Public artifacts contain no participant identifiers, exact private paths, raw values, or original private column names.\n'
 'Candidate discovery does not itself clear official Arm A. Selected companion sources must next undergo semantic review and deterministic binding before FHIR/OMOP enrichment.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase53_shareable_sha256':sha256(P53),
 'input_phase53_resource_counts_sha256':sha256(C53),
 'source_anchor_token':SOURCE_TOKEN,
 'companion_authorization_scope_attested':companion_auth_valid,
 'nearby_files_inspected':len(raw_files),
 'header_companion_candidates':len(candidate_rows),
 'row_level_candidates_profiled':sum(r['profile_status']=='PROFILED' for r in join_rows),
 'dimensions_with_joinable_candidate_evidence':found_count,
 'dimensions_total':len(gap_rows),
 'dimension_candidate_status':{
    r['dimension']:r['candidate_found'] for r in dimension_rows
 },
 'ready_for_official_armA':False,
 'next_step':next_step,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'real_authorized_mri_processed_by_proposed_fidelity_method':False,
 'independently_blinded_external_validation':False,
 'privacy':'PUBLIC_TOKENS_AND_AGGREGATE_OVERLAP_ONLY_NO_IDS_VALUES_OR_PRIVATE_PATHS',
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Joinable companion-source discovery narrows missing evidence but does not clear semantic, vocabulary, official-target, or fidelity-validation gates.'
}
summary_path=STAGE/'ICHI2027_Phase54_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase54_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase54 companion-source audit completed.')
print('Joinable evidence dimensions:',found_count,'/',len(gap_rows))
print('Next step:',next_step)
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase54 companion-source audit completed.
Joinable evidence dimensions: 0 / 6
Next step: COMPANION_EVIDENCE_NOT_FOUND_IN_NEARBY_AUTHORIZED_SOURCE_FAMILY
Shareable: /content/drive/MyDrive/ICHI2027/Phase54/run_20261001T054919_003892Z/ICHI2027_Phase54_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase54/run_20261001T054919_003892Z/ICHI2027_Phase54_PUBLIC_OUTPUTS.zip


## Interpretation after Phase 54

Three outcomes are possible:

- **Several joinable companion dimensions found:** freeze exact field bindings in Phase 55 and enrich the private FHIR source without inventing anything.
- **Only some dimensions found:** enrich only those supported dimensions and keep the remaining gaps explicit.
- **No joinable companion evidence found:** the current authorized source family cannot support an official Arm A as originally specified; report that limitation instead of manufacturing the missing evidence.

Even a successful companion search does not itself constitute an OMOP/MI-CDM load or a real-data fidelity result.
